# PointNet: 3D Point Cloud Classification — Full Walkthrough

This notebook walks through the complete pipeline for 3D object classification using **PointNet** (Qi et al., CVPR 2017) on the **ModelNet10** dataset.

**Contents:**
1. Setup & Dataset Exploration
2. Architecture Explanation
3. Training
4. Evaluation & Confusion Matrix
5. Critical Point Visualization
6. Failure Case Analysis

> **Colab Users:** Run the setup cell below first to install dependencies and clone the repo.

In [ ]:
# === Colab Setup (skip if running locally) ===
# Uncomment the lines below when running on Google Colab

# !pip install torch torchvision open3d numpy matplotlib scikit-learn tqdm trimesh
# !git clone https://github.com/YOUR_USERNAME/pointnet-3d-classification.git
# %cd pointnet-3d-classification

# === Download ModelNet10 ===
# Uncomment if dataset hasn't been downloaded yet:
# !python data/download_modelnet.py

In [ ]:
import sys
import os

# Add project root to path so imports work
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Dataset Exploration

ModelNet10 contains 10 categories of 3D CAD models stored as `.off` mesh files. We sample 1024 points from each mesh surface to create point clouds.

In [ ]:
from data.dataset import ModelNet10Dataset, CLASSES, CLASS_TO_IDX

DATA_ROOT = "data/ModelNet10"  # Adjust if needed

train_dataset = ModelNet10Dataset(root=DATA_ROOT, split="train", num_points=1024, augment=False)
test_dataset = ModelNet10Dataset(root=DATA_ROOT, split="test", num_points=1024, augment=False)

print(f"Classes ({len(CLASSES)}): {CLASSES}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Class distribution
train_counts = [0] * len(CLASSES)
for _, label in train_dataset.samples:
    train_counts[label] += 1

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASSES, train_counts, color="steelblue", edgecolor="white")
for bar, c in zip(bars, train_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(c),
            ha="center", fontsize=9)
ax.set_title("ModelNet10 Training Set — Class Distribution", fontweight="bold")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample point clouds — one per class
fig = plt.figure(figsize=(20, 8))
fig.suptitle("ModelNet10 — Sample Point Clouds (1024 points each)", fontsize=16, fontweight="bold")

class_samples = {}
for i in range(len(train_dataset)):
    points, label = train_dataset[i]
    if label not in class_samples:
        class_samples[label] = points.numpy()
    if len(class_samples) == len(CLASSES):
        break

for idx, (label, points) in enumerate(sorted(class_samples.items())):
    ax = fig.add_subplot(2, 5, idx + 1, projection="3d")
    ax.scatter(points[:, 0], points[:, 1], points[:, 2],
               s=1, c=points[:, 1], cmap="viridis", alpha=0.6)
    ax.set_title(CLASSES[label], fontsize=12)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig("../assets/sample_pointclouds.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to assets/sample_pointclouds.png")

## 2. PointNet Architecture

PointNet's key insight is that a point cloud is an **unordered set** of 3D points. The architecture must be:
- **Permutation invariant** — output shouldn't change if points are reordered
- **Transformation invariant** — should handle rotated/translated inputs

### Architecture Overview

```
Input [B, N, 3]
    │
    ├─ T-Net (3×3) ─── Learns a 3×3 rotation matrix to align input
    │                    to a canonical orientation
    ▼
    Matrix Multiply (align input)
    │
    ├─ Shared MLP (64, 64) ─── Conv1d layers, same weights for every point
    │
    ├─ T-Net (64×64) ─── Aligns learned features (with regularization
    │                      to keep transform near-orthogonal)
    ▼
    Matrix Multiply (align features)
    │
    ├─ Shared MLP (64, 128, 1024) ─── Higher-dimensional per-point features
    │
    ├─ Max Pool ─── THE key operation: aggregates N points into 1 vector
    │               Invariant to point ordering (symmetric function)
    ▼
    Global Feature [B, 1024]
    │
    ├─ FC (512, 256, k) ─── Classification head with dropout
    ▼
    Output [B, k] (class logits)
```

### Why Max Pooling?
Any function on a set must be a **symmetric function** — it shouldn't depend on element ordering. Max pooling is a simple symmetric function: `max({x1, x2, ..., xN})` gives the same result regardless of order. This is PointNet's elegant solution to permutation invariance.

### Why T-Net?
Objects can appear in arbitrary orientations. T-Net learns a transformation matrix that rotates/aligns the input into a canonical pose, making the network more robust to geometric transformations.

In [ ]:
from models.pointnet import PointNetClassifier, TNet, feature_transform_regularization

model = PointNetClassifier(num_classes=10, feature_transform=True)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()

# Test forward pass
dummy = torch.randn(2, 1024, 3)
logits, input_t, feat_t = model(dummy)
print(f"Input shape:             {dummy.shape}")
print(f"Output logits shape:     {logits.shape}")
print(f"Input transform shape:   {input_t.shape}")
print(f"Feature transform shape: {feat_t.shape}")

reg_loss = feature_transform_regularization(feat_t)
print(f"\nFeature regularization loss: {reg_loss.item():.4f}")
print("(Should be close to 0 at convergence — means A*A^T ≈ I)")

## 3. Training

We train PointNet with:
- **Adam** optimizer (lr=0.001, weight_decay=1e-4)
- **StepLR** scheduler (step=20, gamma=0.5)
- **Loss** = CrossEntropy + 0.001 * feature_transform_regularization
- **50 epochs**, batch size 32, 1024 points per sample

Training takes ~15-20 minutes on a T4 GPU.

You can either train here or load a pretrained checkpoint.

In [ ]:
# Option A: Train from scratch (uncomment to run)
# !python train.py --epochs 50 --batch_size 32 --num_points 1024

# Option B: Quick training demo (5 epochs) to see the loop in action
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

TRAIN_DEMO = True  # Set to False to skip and load pretrained

if TRAIN_DEMO:
    train_ds = ModelNet10Dataset(root=DATA_ROOT, split="train", num_points=1024, augment=True)
    test_ds = ModelNet10Dataset(root=DATA_ROOT, split="test", num_points=1024, augment=False)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

    model = PointNetClassifier(num_classes=10, feature_transform=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    demo_epochs = 5
    for epoch in range(1, demo_epochs + 1):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for points, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            points, labels = points.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, _, feat_t = model(points)
            loss = criterion(logits, labels)
            if feat_t is not None:
                loss += 0.001 * feature_transform_regularization(feat_t)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()

        # Quick validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for points, labels in test_loader:
                points, labels = points.to(device), labels.to(device)
                logits, _, _ = model(points)
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        print(f"Epoch {epoch}: Train Loss={running_loss/total:.4f}, "
              f"Train Acc={correct/total:.4f}, Val Acc={val_correct/val_total:.4f}")

    print("\nDemo training complete! For full training, run: python train.py --epochs 50")

## 4. Evaluation & Confusion Matrix

Load the best checkpoint and evaluate on the full test set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load best checkpoint (use the demo model if no checkpoint exists)
checkpoint_path = "../checkpoints/best_model.pth"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    eval_model = PointNetClassifier(num_classes=10, feature_transform=True).to(device)
    eval_model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')}")
else:
    eval_model = model  # Use the demo-trained model
    print("Using demo-trained model (for full results, run train.py first)")

eval_model.eval()

# Run inference on test set
test_ds_eval = ModelNet10Dataset(root=DATA_ROOT, split="test", num_points=1024, augment=False)
eval_loader = DataLoader(test_ds_eval, batch_size=32, shuffle=False, num_workers=0)

all_preds, all_labels = [], []
with torch.no_grad():
    for points, labels in eval_loader:
        points = points.to(device)
        logits, _, _ = eval_model(points)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Overall accuracy
acc = accuracy_score(all_labels, all_preds)
print(f"\nOverall Accuracy: {acc:.4f} ({acc:.1%})")

# Per-class report
print(f"\n{classification_report(all_labels, all_preds, target_names=CLASSES, digits=4)}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=np.arange(len(CLASSES)), yticks=np.arange(len(CLASSES)),
       xticklabels=CLASSES, yticklabels=CLASSES,
       ylabel="True Label", xlabel="Predicted Label",
       title="Confusion Matrix — PointNet on ModelNet10")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.0
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black", fontsize=10)

plt.tight_layout()
plt.savefig("../results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion matrix to results/confusion_matrix.png")

## 5. Critical Point Visualization

**This is the most insightful visualization.** We compute the gradient of the predicted class score with respect to input points. Points with the highest gradient magnitude are the ones that contribute most to the classification — the "critical points" that define the object's identity.

This reveals what PointNet actually learns: it focuses on geometrically distinctive regions (edges, corners, distinctive surface features) rather than flat surfaces.

In [ ]:
def compute_critical_points(model, points_tensor, device, top_k=100):
    """Compute critical points via gradient of predicted class w.r.t. input.

    Returns indices of the top-k most important points.
    """
    model.eval()
    points = points_tensor.unsqueeze(0).to(device).requires_grad_(True)

    logits, _, _ = model(points)
    predicted_class = logits.argmax(dim=1)
    score = logits[0, predicted_class[0]]

    score.backward()

    # Gradient magnitude per point
    grad = points.grad[0]  # [N, 3]
    grad_magnitude = torch.norm(grad, dim=1)  # [N]

    # Top-k critical points
    _, critical_idx = torch.topk(grad_magnitude, top_k)
    return critical_idx.cpu().numpy(), grad_magnitude.cpu().detach().numpy()


# Visualize critical points for several classes
fig = plt.figure(figsize=(20, 8))
fig.suptitle("Critical Points — What PointNet Focuses On\n(Red = high importance, Gray = low importance)",
             fontsize=14, fontweight="bold")

sample_indices = []
seen_classes = set()
for i in range(len(test_ds_eval)):
    _, label = test_ds_eval[i]
    if label not in seen_classes:
        sample_indices.append(i)
        seen_classes.add(label)
    if len(seen_classes) == 10:
        break

for plot_idx, sample_idx in enumerate(sample_indices):
    points, label = test_ds_eval[sample_idx]

    critical_idx, grad_mag = compute_critical_points(eval_model, points, device, top_k=100)
    points_np = points.numpy()

    ax = fig.add_subplot(2, 5, plot_idx + 1, projection="3d")

    # Plot non-critical points in gray
    all_idx = set(range(len(points_np)))
    non_critical = np.array(list(all_idx - set(critical_idx)))
    if len(non_critical) > 0:
        ax.scatter(points_np[non_critical, 0], points_np[non_critical, 1],
                   points_np[non_critical, 2], s=1, c="lightgray", alpha=0.3)

    # Plot critical points in red
    ax.scatter(points_np[critical_idx, 0], points_np[critical_idx, 1],
               points_np[critical_idx, 2], s=8, c="red", alpha=0.9)

    ax.set_title(CLASSES[label], fontsize=11)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig("../results/critical_points.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to results/critical_points.png")

## 6. Failure Case Analysis

Let's examine misclassified samples to understand the model's weaknesses. Common confusions include:
- **night_stand vs dresser** — both are box-shaped furniture with drawers
- **desk vs table** — similar flat surfaces with legs
- **bathtub vs bed** — elongated shapes with raised edges

In [ ]:
# Find misclassified samples
misclassified = []
for i in range(len(all_preds)):
    if all_preds[i] != all_labels[i]:
        misclassified.append(i)

print(f"Total misclassified: {len(misclassified)} / {len(all_preds)}")
print(f"Error rate: {len(misclassified)/len(all_preds):.1%}\n")

# Show up to 5 misclassified examples
num_show = min(5, len(misclassified))
fig = plt.figure(figsize=(4 * num_show, 4))
fig.suptitle("Misclassified Examples", fontsize=14, fontweight="bold")

for plot_idx in range(num_show):
    sample_idx = misclassified[plot_idx]
    points, true_label = test_ds_eval[sample_idx]
    pred_label = all_preds[sample_idx]
    points_np = points.numpy()

    ax = fig.add_subplot(1, num_show, plot_idx + 1, projection="3d")
    ax.scatter(points_np[:, 0], points_np[:, 1], points_np[:, 2],
               s=1, c=points_np[:, 1], cmap="coolwarm", alpha=0.6)
    ax.set_title(f"True: {CLASSES[true_label]}\nPred: {CLASSES[pred_label]}",
                 fontsize=10, color="red")
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig("../results/failure_cases.png", dpi=150, bbox_inches="tight")
plt.show()

# Analyze which class pairs are most confused
print("\nMost confused class pairs:")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if i != j and cm[i, j] >= 2:
            print(f"  {CLASSES[i]:>12} -> {CLASSES[j]:<12} ({cm[i, j]} samples)")

print("\nThese confusions make sense geometrically — the confused classes share")
print("similar global shapes. PointNet lacks local feature extraction, so it")
print("struggles with fine-grained distinctions. PointNet++ addresses this")
print("by adding hierarchical local feature learning.")